In [ ]:
import json
import os
import tempfile
from pathlib import Path

FILE = Path("../complete_dataset_raw/raw_usernames_compressed.jsonl")

CORRECTIONS = {
    "0:c700f30f4761153be3a1e829aa7259434da92d3da5efd05057e279a2c0961a00": {
        "u": "@hacktoluck",
        "c": ["10 Letters"]
    },
    "0:50f33235c6d8c8bda015efae95d1a6e785e38e39c402ccee550cff95cab027b3": {
        "u": "@shopwide",
        "c": ["No_Categories"]
    }
}

counts = {nft: 0 for nft in CORRECTIONS}
temp_path = None

try:
    with tempfile.NamedTemporaryFile(
        mode="w",
        encoding="utf-8",
        dir=FILE.parent,
        prefix=".usernames_",
        suffix=".tmp",
        delete=False
    ) as fout:

        temp_path = Path(fout.name)

        with FILE.open("r", encoding="utf-8") as fin:
            for line in fin:
                if not line.strip():
                    fout.write(line)
                    continue

                record = json.loads(line)
                nft = str(record.get("n", "")).lower()

                if nft in CORRECTIONS:
                    record["u"] = CORRECTIONS[nft]["u"]
                    record["c"] = CORRECTIONS[nft]["c"]
                    counts[nft] += 1

                    line = json.dumps(record, ensure_ascii=False) + "\n"

                fout.write(line)

        fout.flush()
        os.fsync(fout.fileno())

    if all(count > 0 for count in counts.values()):
        os.replace(temp_path, FILE)
        temp_path = None
        print("File originale aggiornato correttamente.")
    else:
        print("Almeno uno dei due NFT non è stato trovato.")
        print("Il file originale non è stato modificato.")

finally:
    if temp_path is not None:
        temp_path.unlink(missing_ok=True)

print("\nActions aggiornate:")
for nft, count in counts.items():
    print(f"{nft}: {count}")

File originale aggiornato correttamente.

Actions aggiornate:
0:c700f30f4761153be3a1e829aa7259434da92d3da5efd05057e279a2c0961a00: 3
0:50f33235c6d8c8bda015efae95d1a6e785e38e39c402ccee550cff95cab027b3: 3


In [2]:
import pandas as pd

trovati = {}

with FILE.open("r", encoding="utf-8") as f:
    for line in f:
        if not line.strip():
            continue

        record = json.loads(line)
        nft = str(record.get("n", "")).lower()

        if nft in CORRECTIONS and nft not in trovati:
            trovati[nft] = {
                "nft": nft,
                "username": record.get("u"),
                "categories": record.get("c")
            }

display(pd.DataFrame(trovati.values()))

,nft,username,categories
0,0:50f33235c6d8c8bda015efae95d1a6e785e38e39c402...,@shopwide,[No_Categories]
1,0:c700f30f4761153be3a1e829aa7259434da92d3da5ef...,@hacktoluck,[10 Letters]
